# 검색광고 · 탐지 모듈 사용법

쇼핑인사이트 쪽은 `get_trend_사용법.ipynb` 에 있다. 여기는 나머지 둘이다.

| 모듈 | 함수 | 주는 것 |
|---|---|---|
| `fashion_trend.searchad` | `get_keywords` | 연관 키워드 + **월간 검색수 절대값** |
| `fashion_trend.detect` | `detect` · `metrics` | 상승률·가속도·지속성 → 점수 → 생애주기 |

아래 출력은 전부 실제로 돌려서 받은 값이다 (2026-10-22). 날짜가 지나면 숫자는 달라진다.

## 준비

`.env` 에 다섯 개가 필요하다. 쇼핑인사이트 두 개, 검색광고 세 개 — **완전히 별개 체계**다.

```
NAVER_HUB_KEY_ID / NAVER_HUB_KEY
NAVER_AD_CUSTOMER_ID / NAVER_AD_API_KEY / NAVER_AD_SECRET_KEY
```

In [2]:
import sys
sys.path.insert(0, "src")

import pandas as pd
from fashion_trend.searchad import get_keywords
from fashion_trend.naver import get_trend
from fashion_trend.detect import detect, metrics

---
# 1부 — 검색광고 키워드도구

## 1-1. 연관어 받기

힌트 키워드를 넣으면 연관어가 쏟아진다. 5개를 넘기면 나눠 호출하고 중복을 없앤다.

In [3]:
kw = get_keywords(["가디건"])

print(len(kw), "개")
kw.head(6)

1200 개


,keyword,pc,mobile,total,comp,masked,hint
0,경량패딩,47100.0,256000.0,303100.0,중간,False,가디건
1,유니클로세일,17100.0,160900.0,178000.0,높음,False,가디건
2,보조배터리,33000.0,140100.0,173100.0,중간,False,가디건
3,후리스,13400.0,82800.0,96200.0,높음,False,가디건
4,슬리퍼,18900.0,73600.0,92500.0,높음,False,가디건
5,패딩,12500.0,56500.0,69000.0,높음,False,가디건


```
1200 개

  keyword       pc    mobile     total comp  masked    hint
0    경량패딩  29500.0  164100.0  193600.0   중간   False  가디건,니트
1   보조배터리  31100.0  139000.0  170100.0   중간   False  가디건,니트
2  유니클로세일  13700.0  145000.0  158700.0   높음   False  가디건,니트
3    암막커튼  10200.0   56500.0   66700.0   중간   False  가디건,니트
4     후리스   8960.0   54100.0   63060.0   높음   False  가디건,니트
5      패딩   9210.0   40300.0   49510.0   높음   False  가디건,니트
```

힌트 2개로 **1,200개**가 왔다. 열의 뜻은 이렇다.

| 열 | 뜻 |
|---|---|
| `pc` · `mobile` | 월간 검색수 **절대값**. 요청이 달라도 그대로 비교된다 |
| `total` | 둘의 합 |
| `comp` | 광고 경쟁 정도 (높음/중간/낮음) |
| `masked` | 네이버가 값을 가렸는지 (아래 1-2) |
| `hint` | 어느 힌트에서 나온 연관어인지 |

쇼핑인사이트와 달리 **앵커도 환산도 필요 없다.** 37,640 은 그냥 37,640 이다.

## 1-2. 가려진 값 — `masked`

월 10회 미만이면 네이버가 숫자 대신 `"< 10"` 이라는 **문자열**을 준다.
그대로 `int()` 하면 터진다. 그래서 NaN 으로 두고 `masked` 플래그를 세웠다.

In [ ]:
print("가려진 행:", int(kw["masked"].sum()), "/", len(kw))
kw[kw["masked"]].head(4)[["keyword", "pc", "mobile", "total", "comp"]]

```
가려진 행: 246 / 1200

      keyword  pc  mobile  total comp
620  노르딕패턴가디건 NaN   290.0  290.0   높음
720    흰색니트코디 NaN   200.0  200.0   높음
768  화이트니트가디건 NaN   160.0  160.0   낮음
770  바닐라니트가디건 NaN   160.0  160.0   높음
```

1,200개 중 246개(20%)가 한쪽 이상 가려졌다. PC 쪽이 주로 가려지고 모바일은 남는다.

**이게 조기 탐지에 아프다.** 가려지는 구간이 바로 신생 키워드가 사는 구간이다.
쇼핑인사이트의 `0.00437` 양자화 바닥과 같은 성격의 문제다.

## 1-3. 연관어는 생각보다 느슨하다  ★

위 출력을 다시 보자. 상위에 **보조배터리 · 암막커튼** 이 있다. 패션이 아니다.

네이버가 주는 건 "의미가 비슷한 키워드" 가 아니라 **"같이 광고하기 좋은 키워드"** 다.
광고 도구이기 때문이다. 그래서 발굴 결과를 그대로 쓰면 안 되고 반드시 걸러야 한다.

In [ ]:
# 규모 하한 + 가려지지 않은 것만
후보 = kw[(kw["total"] >= 1000) & (~kw["masked"])]
print("월 1,000회 이상:", len(후보), "개")

# 패션이 아닌 것은 여기서 걸러진다 — 지금은 단어 기준, 나중엔 LLM 분류
제외 = ["배터리", "커튼", "캠핑", "세일", "쿠폰", "중고", "남성", "남자", "아동"]
후보 = 후보[~후보["keyword"].str.contains("|".join(제외))]
print("패턴 제외 후:", len(후보), "개")
후보.head(10)[["keyword", "total", "comp"]]

`config/keywords.yaml` 의 `exclude_patterns` 가 이 역할인데 지금 11개뿐이라 한참 부족하다.
결국 단어 매칭이 아니라 **LLM 속성 분류**로 넘어가야 하는 자리다.

---
# 2부 — 탐지 지표

## 2-1. 한 줄로 돌리기

`detect` 는 지표 계산 → Z 표준화 → 점수 → 생애주기 분류를 한 번에 한다.
계절성을 보려면 1년 이상이 필요하므로 78주를 받는다.

In [ ]:
df = get_trend(["원피스", "가디건", "니트", "코트", "패딩"],
               weeks=78, gender="f", ages=["10", "20"])

r = detect(df)
r[["level", "growth", "accel", "persistence", "snr",
   "n_obs", "n_missing", "score", "stage", "direction"]].round(3)

```
         level  growth  accel  persistence    snr  n_obs  n_missing  score      stage direction
패딩       1.468   1.424  0.877         0.75 -0.552     71          8  1.096   Emerging     watch
니트       3.377   1.486 -4.241         0.75 -0.207     78          1  0.294   Emerging     watch
코트       3.249  -0.157 -1.257         0.50 -0.463     78          1 -0.284  Declining      stop
원피스      8.792  -0.512 -1.076         0.50 -0.809     79          0 -0.406  Declining      stop
가디건      4.038   0.023 -2.093         0.25 -0.570     79          0 -0.701     Mature      hold
```

점수 내림차순으로 정렬된다. 10월 기준 패딩·니트가 오르고 원피스가 꺾이는 모양인데,
계절을 생각하면 그럴듯하다.

## 2-2. 열의 뜻

| 열 | 정의 | 왜 보는가 |
|---|---|---|
| `level` | 최근 4주 평균 지수 | 규모. 너무 작으면 지표를 믿을 수 없다 |
| `growth` | 최근 4주 ÷ 직전 4주 − 1 | 성장 |
| `accel` | 상승률의 변화 | 초기 급등 포착 |
| `persistence` | 최근 4주 중 오른 주의 비율 | 일회성 스파이크 제외 |
| `novelty` | 최근 수준 ÷ 장기 평균 − 1 | 처음 보는 수준인가 |
| `snr` | 상승폭 ÷ 과거 변동성 | 원래 요동치는 키워드 걸러내기 |
| `n_missing` | 네이버가 안 보낸 주 수 | 많으면 지표가 불안정 |
| `at_floor` | 양자화 바닥 근처인가 | True 면 수치를 믿지 말 것 |
| `sparse` | 결측 30% 초과 | 같은 뜻의 경고 |

`stage` 는 Emerging / Growing / Mature / Flat / Declining,
`direction` 은 watch / hold / **stop** 이다.
설계서 §5-5 대로 **하락 경보를 상승 추천과 동등하게** 낸다 — 잘못 떼오면 손실이 크고,
못 떼오면 기회손실인데 작기 때문이다.

## 2-3. `yoy` 는 지금 쓰면 안 된다  ★

전년 동기 대비도 계산은 하는데, 결과를 보면 **전부 음수**다.

In [ ]:
metrics(df, window=4)[["level", "growth", "yoy", "at_floor", "sparse"]].round(3)

```
         level  growth    yoy  at_floor  sparse
가디건      4.038   0.023 -0.781     False   False
니트       3.377   1.486 -0.624     False   False
원피스      8.792  -0.512 -0.688     False   False
코트       3.249  -0.157 -0.714     False   False
패딩       1.468   1.424 -0.882     False   False
```

다섯 개가 전부 작년보다 62~88% 낮다. 키워드 문제가 아니다.

확인해보니 **네이버쇼핑 클릭이 전 카테고리에서 1년 사이 35~44% 줄었다.**
식품·가전·가구까지 같은 비율이라 소비자 취향 변화가 아니라 플랫폼 쪽 변화다.
10~20대 여성 패션은 −80% 로, 공통분 위에 추가 하락이 얹혀 있다.

**그래서 `yoy` 는 분야 전체로 나눠 보정해야 한다.** 아직 구현 안 됐다.
날것으로 쓰면 모든 키워드가 영원히 하락으로 나온다.

## 2-4. 가중치는 임시값이다

In [ ]:
from fashion_trend.detect import WEIGHTS
print(WEIGHTS)

# 바꿔서 돌려볼 수 있다
detect(df, weights={"growth": 0.6, "accel": 0.2, "persistence": 0.2})[["score", "stage"]].round(3)

`score = 0.4·Z(growth) + 0.3·Z(accel) + 0.3·Z(persistence)` 인데
**이 숫자에 근거는 없다.** 설계서 §5-2 가 명시한 대로, 과거 시점에서 추천한 뒤
실제로 이후 4~8주 올랐는지 백테스트해서 정해야 한다. 그때까지 순위를 그대로 믿지 않는다.

---
# 3부 — 두 소스 합치기

쇼핑인사이트는 **움직임**을, 검색광고는 **규모**를 준다. 서로 없는 걸 메운다.

| | 쇼핑인사이트 | 검색광고 |
|---|---|---|
| 값 | 상대지수 | 절대 검색수 |
| 시간 | 주/일 단위 시계열 | 최근 30일 한 줄 |
| 세그먼트 | 성별·연령 분해 | 없음 |
| 용도 | 상승률·가속도 | 규모 하한 |

In [ ]:
size = (get_keywords(["가디건", "니트", "코트", "패딩", "원피스"])
        .set_index("keyword")["total"])

merged = (detect(df)[["level", "growth", "persistence", "score", "stage"]]
          .join(size.rename("월간검색수")))
merged.round(3)

규모가 붙으면 **"상승률은 높은데 월 200회짜리"** 같은 후보를 걸러낼 수 있다.
증감률만 보면 월 50회가 150회 된 것도 300% 상승이라 1위로 올라온다.

---
## 기억할 것

1. 연관어에는 패션이 아닌 것이 섞여 있다. **반드시 거른다**
2. `< 10` 은 문자열이다. `masked` 가 True 면 그 수치는 없는 것으로 친다
3. `at_floor` · `sparse` 가 True 면 지표를 믿지 않는다
4. `yoy` 는 분야 보정 전까지 쓰지 않는다
5. `score` 의 가중치는 백테스트 전까지 임시값이다